# STAGE 6B · Validation fixes

Two defects in the Stage 6 run, both found by reading its own output.

### ❌ Fix 1 — the negative control was void

It printed `✅ control collapses`, but the pass condition was
`abs(fake) < abs(real)/2 or real <= 0`, and `real = -0.0008 <= 0`, so it auto-passed. It also tested **AUROC**, which ACR cannot move by construction — a monotone recalibration cannot reorder cases within a stratum.

Replaced with a **four-arm ablation**:

| arm | what |
|---|---|
| **A raw** | the classifier as shipped |
| **B platt** | recalibrate on the classifier logit ONLY ← **the real null** |
| **C acr** | + real acquisition covariates |
| **D shuffled** | + row-shuffled acquisition covariates |

> Even with acquisition destroyed, the fit still contains `β₀ + β₁·logit(p̂)` — ordinary Platt scaling, which improves ECE on its own. **The claim requires C to beat B, not merely to beat A.**

The verdict turns on mean **within-group** ECE, not `|ECE_AP − ECE_PA|`. The gap alone is gameable: a model equally badly calibrated in *both* groups scores a perfect 0.

### ⚠️ Fix 2 — the AP/PA gap needs a CI

It is now the headline result (−0.064 AUROC, 8/8 pathologies) and had only my informal sign test. Adds a stratified bootstrap per pathology, plus a **cluster bootstrap over films** for the mean — the 8 labels sit on the same radiographs and are strongly correlated, so treating them as independent would understate the variance.

---
**CPU only. ~3 minutes. Zero compute units.**

---
# 0 · Config

**Colab CPU runtime costs 0 compute units** — units are billed for GPU/TPU only.

Reads everything straight off Drive, written by your Stage 6 full run:

```
MyDrive/Component_01/reports/stage6/cache/
    probs_val.npy      probs_test.npy       <- classifier outputs
    imgfeat_val.parquet  imgfeat_test.parquet <- acquisition proxies
```

> Because the image features were cached, **the tar does not need extracting** and no image is opened. Set `ON_COLAB = False` to run on your laptop instead.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

ON_COLAB = True          # True -> Drive (CPU runtime, 0 compute units)

if ON_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('  Drive already mounted')
    PROJECT  = Path('/content/drive/MyDrive/Component_01')
    METADATA = PROJECT / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
    IMG_ROOT = Path('/content/cardio_image_384')   # only needed if caches are absent
else:
    HERE     = Path.cwd()
    PROJECT  = HERE if (HERE / 'training_manifest').exists() else HERE / 'Component_01'
    METADATA = PROJECT.parent / 'data' / 'raw' / 'mimic-cxr-2.0.0-metadata.csv'
    IMG_ROOT = PROJECT.parent / 'data' / 'output' / 'cardio_image_384'

MANIFEST = PROJECT / 'training_manifest'
OUT      = PROJECT / 'reports' / 'stage6'; OUT.mkdir(parents=True, exist_ok=True)
CACHE    = OUT / 'cache'; CACHE.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT))
import stage6_acr as acr
PATH = acr.PATHOLOGIES

print('  PROJECT', PROJECT)
print()
need = True
for s in ('val', 'test'):
    f = CACHE / ('probs_' + s + '.npy')
    print(('  OK      ' if f.exists() else '  MISSING '), f.name)
    need &= f.exists()
assert need, ('probs_val.npy / probs_test.npy not found in ' + str(CACHE) +
              ' -- these are written by the Stage 6 full run.')

# Stage 6 also cached the image features. If they are here, this notebook
# never touches an image and the tar does NOT need extracting.
HAVE_IMGFEAT = all((CACHE / ('imgfeat_' + s + '.parquet')).exists()
                   for s in ('val', 'test'))
for s in ('val', 'test'):
    f = CACHE / ('imgfeat_' + s + '.parquet')
    print(('  OK      ' if f.exists() else '  absent  '), f.name)
print()
print('  image features cached:', HAVE_IMGFEAT,
      '-> no tar extraction needed' if HAVE_IMGFEAT else '-> images required (see below)')

n_pass, n_fail = acr._selftest(verbose=False)
assert n_fail == 0, 'method self-test failed'
print('  method self-test:', n_pass, 'passed')

---
# 1 · Rebuild the acquisition table

DICOM covariates are re-derived (instant); pixel proxies are reused from the Stage 6 cache.

> Stage 6 saved `acquisition_test.csv` but **not** the val table — and refitting every ablation arm needs it. The row-count assert catches the classic trap of a leftover 300-row *smoke* cache.

In [ ]:
man  = {s: pd.read_csv(MANIFEST / ('manifest_' + s + '.csv'), low_memory=False)
        for s in ('val', 'test')}
meta = pd.read_csv(METADATA, low_memory=False)

acq = {}
for s, d in man.items():
    a = acr.metadata_acquisition(d, meta)          # exact, from DICOM headers
    cached = CACHE / ('imgfeat_' + s + '.parquet')          # written by Stage 6
    local  = CACHE / ('imgfeat_local_' + s + '.parquet')    # written by this file
    if cached.exists():
        imf = pd.read_parquet(cached)
        print('  ' + s + ': image features reused from the Stage 6 run')
    elif local.exists():
        imf = pd.read_parquet(local)
        print('  ' + s + ': image features from local cache')
    else:
        assert IMG_ROOT.exists(), (
            'No cached image features AND no images at ' + str(IMG_ROOT) + '.\n'
            'On Colab: extract cardio_384.tar to /content first, or copy\n'
            'imgfeat_val.parquet + imgfeat_test.parquet into ' + str(CACHE))
        t0 = time.time()
        imf = acr.image_acquisition_batch(
            [str(IMG_ROOT / p) for p in d.image_path],
            progress=lambda i, n: print('\r  %s: %d/%d  %.0fs left'
                % (s, i, n, (time.time()-t0)/i*(n-i)), end=''))
        imf.to_parquet(local)
        print('\r  %s: %d images in %.0fs' % (s, len(imf), time.time()-t0) + ' '*20)
    assert len(imf) == len(d), (
        s + ': cached image features have ' + str(len(imf)) + ' rows but the '
        'manifest has ' + str(len(d)) + '. The cache is from a SMOKE run -- '
        'delete imgfeat_' + s + '.parquet and re-run Stage 6 with SMOKE_TEST=False.')
    acq[s] = pd.concat([a.reset_index(drop=True), imf.reset_index(drop=True)], axis=1)
    acq[s]['stratum'] = acr.stratum_id(acq[s])

probs  = {s: pd.DataFrame(np.load(CACHE / ('probs_' + s + '.npy')), columns=PATH)
          for s in ('val', 'test')}
labels = {s: man[s][PATH].astype(int).reset_index(drop=True) for s in ('val', 'test')}
for s in ('val', 'test'):
    assert len(probs[s]) == len(labels[s]) == len(acq[s]), (
        'row mismatch in ' + s + ': probs=' + str(len(probs[s])) +
        ' labels=' + str(len(labels[s])) + ' acq=' + str(len(acq[s])))

_m = np.nanmean([acr.auroc(labels['test'][k], probs['test'][k]) for k in PATH])
print('\n  mean AUROC on test: %.4f' % _m, '  (Stage 6 reported 0.8554)')
assert abs(_m - 0.8554) < 0.01, 'cached probabilities do not match the Stage 6 run'

---
# 2 · FIX 1 — four-arm calibration ablation ★

**The decisive column is `group ECE`.** If arm C does not beat arm B, the contribution reduces to *"we applied Platt scaling"* and must be reported that way.

In [ ]:
AB = acr.calibration_ablation(probs['val'], labels['val'], acq['val'],
                              probs['test'], labels['test'], acq['test'], PATH)

print('=' * 88)
print('  FOUR-ARM CALIBRATION ABLATION  (test set, n=%d)' % len(labels['test']))
print('=' * 88)
print('  %-34s %9s %9s %11s %11s' % ('arm', 'AUROC', 'ECE', 'group ECE', 'subgrp gap'))
print('  ' + '-' * 85)
NAMES = {'A_raw': 'A  raw classifier',
         'B_platt': 'B  Platt only (logit p-hat)  <- NULL',
         'C_acr': 'C  ACR (real acquisition)',
         'D_shuffled': 'D  ACR (shuffled acquisition)'}
for t in ('A_raw', 'B_platt', 'C_acr', 'D_shuffled'):
    r = AB[t]
    print('  %-34s %9.4f %9.4f %11.4f %11.4f'
          % (NAMES[t], r['mean_auroc'], r['mean_ece'],
             r['mean_group_ece'], r['mean_subgroup_gap']))
print('  ' + '-' * 85)
v = AB['verdict']
print()
print('  C vs B  group-ECE gain : %+.4f' % v['group_ece_gain_over_platt'])
print('  C vs B  subgroup gain  : %+.4f' % v['subgroup_gain_over_platt'])
print()
if v['acquisition_matters']:
    print('  ACQUISITION ADDS REAL SIGNAL BEYOND PLATT SCALING.')
    print('  Arm D (shuffled) must sit near arm B -- check the table above.')
else:
    print('  ACQUISITION ADDS NOTHING BEYOND PLATT SCALING.')
    print('  Report the calibration result as ordinary recalibration, NOT as an')
    print('  acquisition-conditioned contribution. Do not claim otherwise.')

print()
print('  per-pathology group ECE')
print('  %-20s %9s %9s %9s %9s' % ('pathology', 'raw', 'platt', 'ACR', 'shuffled'))
print('  ' + '-' * 60)
for k in PATH:
    print('  %-20s %9.4f %9.4f %9.4f %9.4f'
          % (k, AB['A_raw']['per_pathology'][k]['group_ece'],
             AB['B_platt']['per_pathology'][k]['group_ece'],
             AB['C_acr']['per_pathology'][k]['group_ece'],
             AB['D_shuffled']['per_pathology'][k]['group_ece']))

---
# 3 · FIX 2 — AP/PA gap with confidence intervals ★

Your headline finding. Per-pathology CIs use a **stratified** bootstrap; the mean uses a **cluster** bootstrap over films.

In [ ]:
ap = (acq['test'].is_AP > 0.5).to_numpy()
print('  AP films %d   PA films %d' % (ap.sum(), (~ap).sum()))
print('  bootstrapping (1000 reps, ~30s) ...')
GT = acr.projection_gap_test(labels['test'], probs['test'], ap, PATH, n_boot=1000)

print()
print('=' * 88)
print('  PA-minus-AP AUROC GAP  (positive => classifier is WORSE on AP films)')
print('=' * 88)
print('  %-20s %9s %9s %9s %22s' % ('pathology', 'AP', 'PA', 'gap', '95% CI'))
print('  ' + '-' * 85)
for k in PATH:
    r = GT['per_pathology'][k]
    sig = 'YES' if r['lo'] > 0 else ('no ' if r['hi'] > 0 else 'REV')
    print('  %-20s %9.4f %9.4f %+9.4f   [%+.4f, %+.4f]  %s'
          % (k, r['ap'], r['pa'], r['gap'], r['lo'], r['hi'], sig))
print('  ' + '-' * 85)
print('  %-20s %29.4f   [%+.4f, %+.4f]'
      % ('MEAN (cluster boot)', GT['mean_gap'], GT['mean_lo'], GT['mean_hi']))
print()
print('  pathologies favouring PA : %d / %d' % (GT['n_favouring_pa'], GT['n_pathologies']))
print('  bootstrap P(mean gap <= 0): %.4f' % GT['mean_p_gt_0'])
print()
if GT['mean_lo'] > 0:
    print('  CONFIRMED: the classifier is significantly worse on AP films.')
    print('  AP films come disproportionately from the sickest, bedridden,')
    print('  emergency patients -- and pooled evaluation hides this entirely.')
else:
    print('  NOT SIGNIFICANT: the CI includes zero. Do not claim an AP/PA gap.')

---
# 4 · Save

In [ ]:
from datetime import datetime
res = dict(stage='6b', timestamp=datetime.now().isoformat(),
           n_test=int(len(labels['test'])), n_ap=int(ap.sum()), n_pa=int((~ap).sum()),
           calibration_ablation={t: {kk: vv for kk, vv in AB[t].items()}
                                 for t in ('A_raw','B_platt','C_acr','D_shuffled')},
           verdict=AB['verdict'], projection_gap=GT)
(OUT / 'stage6b_validation.json').write_text(
    json.dumps(res, indent=2, default=float), encoding='utf-8')
print('  saved', OUT / 'stage6b_validation.json')

---
# What this settles

| question | answered by |
|---|---|
| Does acquisition add anything beyond Platt scaling? | §2, arm **C vs B** |
| Is the shuffled control actually null? | §2, arm **D ≈ B** |
| Is the AP/PA gap statistically real? | §3, cluster-bootstrap CI |

**If §2 says acquisition adds nothing, say so.** The AP/PA performance gap (§3) stands on its own as a finding regardless — it is measured directly from the classifier and does not depend on ACR working.